## Now Creating Fact table with primary key and foreign keys

In [0]:
#importing libraries
#importing necessary libraryfrom pyspark.sql import SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import upper, col,to_date, date_format, broadcast, udf, monotonically_increasing_id
from pyspark.sql.types import DoubleType

In [0]:
spark = SparkSession.builder.appName("factSales").getOrCreate()

In [0]:
#directory lookup
dbutils.fs.ls("/FileStore/tables/")

Out[19]: [FileInfo(path='dbfs:/FileStore/tables/BigMart_Sales-1.csv', name='BigMart_Sales-1.csv', size=869537, modificationTime=1749763446000),
 FileInfo(path='dbfs:/FileStore/tables/BigMart_Sales-2.csv', name='BigMart_Sales-2.csv', size=869537, modificationTime=1749763573000),
 FileInfo(path='dbfs:/FileStore/tables/BigMart_Sales.csv', name='BigMart_Sales.csv', size=869537, modificationTime=1749763408000),
 FileInfo(path='dbfs:/FileStore/tables/DIM_CustomerSegment/', name='DIM_CustomerSegment/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/DIM_Date/', name='DIM_Date/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/DIM_Employee/', name='DIM_Employee/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/DIM_Product/', name='DIM_Product/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/DIM_ProductCategory/', name='DIM_ProductCategory/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/DIM

In [0]:
#importing  delta file of preproced csv and dim tables

#delta file of original csv("Preprocessed null valued file")
fact_sales = spark.read.format("delta").load("/FileStore/tables/Fact_Sales_Preprocessed")
dim_product = spark.read.format("delta").load("/FileStore/tables/DIM_Product")
dim_store = spark.read.format("delta").load("/FileStore/tables/DIM_Store")
dim_employee = spark.read.format("delta").load("/FileStore/tables/DIM_Employee")



In [0]:
fact_sales_joined = fact_sales \
    .join(
        broadcast(dim_product.select("Product_Id", "ProductCategory", "ProductName", "Brand")),
        on=["ProductCategory", "ProductName", "Brand"],
        how="left"
    ) \
    .join(
        broadcast(dim_store.select("Store_Id", "StoreRegion", "StoreName", "StoreType")),
        on=["StoreRegion", "StoreName", "StoreType"],
        how="left"
    ) \
    .join(
        broadcast(dim_employee.select("Employee_Id", "SalesRep", "Department", "EmployeeRole")),
        on=["SalesRep", "Department", "EmployeeRole"],
        how="left"
    )


In [0]:
fact_sales_joined.display()

SalesRep,Department,EmployeeRole,StoreRegion,StoreName,StoreType,ProductCategory,ProductName,Brand,UnitsSold,UnitPrice,Discount,SaleDate,Product_Id,Store_Id,Employee_Id
Martha Long,Electronics,Cashier,East,StoreX,Franchise,Furniture,T-shirt,BrandB,12.0,-1.0,5.0,2022-12-14,23,8,22
Martha Long,Home,Sales Associate,East,StoreZ,Franchise,Clothing,Tablet,BrandC,-1.0,272.49,-1.0,2023-02-24,22,10,12
Emily Vazquez,Apparel,Cashier,South,StoreX,Retail,Clothing,Tablet,BrandA,-1.0,484.75,15.0,2025-03-24,5,19,15
Charles Fields,Apparel,Cashier,West,StoreY,Outlet,Electronics,Smartphone,BrandB,-1.0,205.74,10.0,2023-09-30,2,13,14
Wendy Castillo,Home,Manager,East,StoreZ,Outlet,Furniture,T-shirt,BrandC,46.0,20.25,5.0,2022-10-14,25,20,3
Wendy Castillo,Home,Manager,South,StoreY,Retail,Furniture,T-shirt,BrandC,-1.0,361.06,10.0,2024-02-23,25,2,3
John Harris,Home,Cashier,South,StoreY,Outlet,Clothing,T-shirt,BrandC,37.0,492.65,5.0,2024-05-06,11,16,10
Charles Fields,Home,Sales Associate,South,StoreX,Outlet,Electronics,Smartphone,BrandC,37.0,293.87,15.0,2023-04-04,1,9,23
Wendy Castillo,Electronics,Manager,South,StoreY,Retail,Clothing,Jeans,BrandA,23.0,189.47,15.0,2022-12-26,19,2,4
Charles Fields,Apparel,Manager,East,StoreZ,Franchise,Furniture,T-shirt,BrandB,25.0,359.08,10.0,2022-10-28,23,10,2


In [0]:
fact_sales_final = fact_sales_joined.select("Product_Id","Store_Id","Employee_Id","UnitsSold", "UnitPrice", "Discount", "SaleDate")

In [0]:
fact_sales_final.display()

Product_Id,Store_Id,Employee_Id,UnitsSold,UnitPrice,Discount,SaleDate
23,8,22,12.0,-1.0,5.0,2022-12-14
22,10,12,-1.0,272.49,-1.0,2023-02-24
5,19,15,-1.0,484.75,15.0,2025-03-24
2,13,14,-1.0,205.74,10.0,2023-09-30
25,20,3,46.0,20.25,5.0,2022-10-14
25,2,3,-1.0,361.06,10.0,2024-02-23
11,16,10,37.0,492.65,5.0,2024-05-06
1,9,23,37.0,293.87,15.0,2023-04-04
19,2,4,23.0,189.47,15.0,2022-12-26
23,10,2,25.0,359.08,10.0,2022-10-28


In [0]:
#function for calulating netRevenue

def netRev(UnitsSold, UnitPrice, Discount):
    if UnitsSold == -1 or UnitPrice == -1 or Discount == -1:
        return "-1"
    return UnitsSold * UnitPrice * (1 - Discount / 100)


In [0]:
#using udf
netRev_udf = udf(netRev, DoubleType())
fact_sales_final_table= fact_sales_final.withColumn("NetRevenue", (netRev_udf("UnitsSold", "UnitPrice", "Discount") ))
#hadeling null values if netRevenue if available with -1
fact_sales_final_table = fact_sales_final_table.fillna({
    "NetRevenue": -1
})

In [0]:
#adding primary key to fact table
fact_sales_final_table_withId= fact_sales_final_table.withColumn("Sales_Id", monotonically_increasing_id()+1)

In [0]:
fact_sales_final_table_withId.display()

Product_Id,Store_Id,Employee_Id,UnitsSold,UnitPrice,Discount,SaleDate,NetRevenue,Sales_Id
23,8,22,12.0,-1.0,5.0,2022-12-14,-1.0,1
22,10,12,-1.0,272.49,-1.0,2023-02-24,-1.0,2
5,19,15,-1.0,484.75,15.0,2025-03-24,-1.0,3
2,13,14,-1.0,205.74,10.0,2023-09-30,-1.0,4
25,20,3,46.0,20.25,5.0,2022-10-14,884.925,5
25,2,3,-1.0,361.06,10.0,2024-02-23,-1.0,6
11,16,10,37.0,492.65,5.0,2024-05-06,17316.6475,7
1,9,23,37.0,293.87,15.0,2023-04-04,9242.2115,8
19,2,4,23.0,189.47,15.0,2022-12-26,3704.1385,9
23,10,2,25.0,359.08,10.0,2022-10-28,8079.3,10


In [0]:
fact_sales_final_table_withId.write \
    .format("delta")\
        .mode("overwrite")\
            .option("mergeSchema", True) \
                .save("/FileStore/tables/FACT_Sales_day25")

### Data validation: CheckallIDs in FACT join with DIM or not (Within same notebook as fact)

In [0]:
data_validation1 = fact_sales_final_table\
  .join(broadcast(dim_product), on= "Product_Id", how="left" ) \
    .join(broadcast(dim_employee), on = "Employee_Id", how = "left")\
      .join(broadcast(dim_store), on = "Store_Id", how="left")

data_validation1.display()

Store_Id,Employee_Id,Product_Id,UnitsSold,UnitPrice,Discount,SaleDate,NetRevenue,ProductCategory,ProductName,Brand,SalesRep,Department,EmployeeRole,StoreRegion,StoreName,StoreType
8,22,23,12.0,-1.0,5.0,2022-12-14,-1.0,Furniture,T-shirt,BrandB,Martha Long,Electronics,Cashier,East,StoreX,Franchise
10,12,22,-1.0,272.49,-1.0,2023-02-24,-1.0,Clothing,Tablet,BrandC,Martha Long,Home,Sales Associate,East,StoreZ,Franchise
19,15,5,-1.0,484.75,15.0,2025-03-24,-1.0,Clothing,Tablet,BrandA,Emily Vazquez,Apparel,Cashier,South,StoreX,Retail
13,14,2,-1.0,205.74,10.0,2023-09-30,-1.0,Electronics,Smartphone,BrandB,Charles Fields,Apparel,Cashier,West,StoreY,Outlet
20,3,25,46.0,20.25,5.0,2022-10-14,884.925,Furniture,T-shirt,BrandC,Wendy Castillo,Home,Manager,East,StoreZ,Outlet
2,3,25,-1.0,361.06,10.0,2024-02-23,-1.0,Furniture,T-shirt,BrandC,Wendy Castillo,Home,Manager,South,StoreY,Retail
16,10,11,37.0,492.65,5.0,2024-05-06,17316.6475,Clothing,T-shirt,BrandC,John Harris,Home,Cashier,South,StoreY,Outlet
9,23,1,37.0,293.87,15.0,2023-04-04,9242.2115,Electronics,Smartphone,BrandC,Charles Fields,Home,Sales Associate,South,StoreX,Outlet
2,4,19,23.0,189.47,15.0,2022-12-26,3704.1385,Clothing,Jeans,BrandA,Wendy Castillo,Electronics,Manager,South,StoreY,Retail
10,2,23,25.0,359.08,10.0,2022-10-28,8079.3,Furniture,T-shirt,BrandB,Charles Fields,Apparel,Manager,East,StoreZ,Franchise
